# Capítulo 6 — Evaluar un modelo y medir la incertidumbre · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch06_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin producir errores. La solución está en `ch06_corrige.ipynb`, en la misma página complementaria.

**Índice.** Datos · Los cálculos de las lecciones 6.1 a 6.6 · Ejercicios 6.1 a 6.6

**El protocolo, en todo el notebook:** ajustar en el **entrenamiento**, elegir en la **validación**, evaluar **una sola vez** en la **prueba**. Para una prueba A/B: comparar dos grupos **asignados al azar**, y dar la diferencia **con su intervalo de confianza**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color por conjunto
SET_TRAIN, SET_VAL, SET_TEST = '#2e7d32', '#b84d00', '#1565c0'   # entrenamiento (verde), validación (naranja oscuro), prueba (azul)
SPAM, LEGIT = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, RESIDUAL, THRESHOLD = '#757575', '#00897b', '#c2185b', '#8e24aa'   # prueba A/B: A gris, B verde azulado


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def mean_loss(labels, X, w, b):
    """Pérdida logarítmica media (ecuaciones 5.6 y 5.7)."""
    p = np.clip(sigmoid(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(labels, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


FEATURE_NAMES = ['links', 'suspicious_words', 'length', 'capitals', 'hour', 'attachments', 'size', 'recipients']


def make_messages(seed=77):
    """Mensajes ficticios (ejercicio 6.4 y lección 6.4): 40 grupos, campañas de spam (grupos pares) o
    remitentes legítimos (grupos impares). Los mensajes de un grupo son casi idénticos. Solo las dos
    primeras características dependen de la etiqueta; las ocho están en unidades muy distintas."""
    rng = np.random.default_rng(seed)
    units = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    offsets = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    rows = []
    for g in range(40):
        spam = g % 2 == 0
        size = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(size):
            rows.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * units + offsets))
    return pd.DataFrame(rows, columns=['group', 'spam'] + FEATURE_NAMES)


def fit(X, y, lam, l1=False, step_k=0.1, n_steps=3000):
    """Descenso de gradiente sobre la pérdida penalizada: L2 (ecuación 6.2) o L1 (ecuación 6.3). b no se penaliza."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_steps):
        gap = sigmoid(X @ w + b) - y
        gw, gb = X.T @ gap / len(y), gap.mean()
        if l1:   # paso de gradiente, luego acercamos cada peso a 0 como mucho en pas × λ (sin cambiarle el signo)
            w = w - step_k * gw
            w = np.sign(w) * np.maximum(np.abs(w) - step_k * lam, 0)
        else:    # la derivada de λ × suma de los w² es 2λw
            w = w - step_k * (gw + 2 * lam * w)
        b -= step_k * gb
    return w, b

## Datos

Los identificadores D6.1 a D6.5, D6.E1, D6.E2, D6.E3, D6.E5 y D5.1 son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 6, copiados de los archivos CSV del libro (no modificar)

# D6.1 — los tres conjuntos de la asociación: mensajes, spams y errores del filtro para un umbral
subsets = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.2 — dos divisiones de los mismos mensajes: el conjunto de prueba, con umbral 0,7
leak = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['split', 'messages', 'spams', 'copies_seen', 'missed_copies', 'missed_new', 'fp'])

# D6.3 — seis mensajes nuevos (validación de la lección 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D5.1 — los doce mensajes del capítulo 5 (entrenamiento de la lección 6.3)
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D6.E2 — la mediateca (ejercicio 6.2): conjuntos de validación y de prueba
media_library = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.4 — los tres pliegues de la asociación (lección 6.4): enlaces (media, desviación típica) y pérdidas de validación
folds = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['fold', 'messages', 'spams', 'mean_links', 'std_links', 'loss_l001', 'loss_l01'])

# D6.5 — la prueba A/B de la librería (lecciones 6.5 y 6.6)
bookshop = pd.DataFrame([
    ('A', 'carta actual', 2500, 200),
    ('B', 'envío gratuito', 2400, 240)
], columns=['version', 'description', 'customers', 'orders'])

# D6.E1 — validación cruzada de la mediateca (ejercicio 6.1): pérdida de cada pliegue para cinco λ
folds_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['fold', 'lambda', 'loss'])

# D6.E3 — reparto de cuatro grupos en los pliegues del voluntario (ejercicio 6.3)
folds_of_groups = pd.DataFrame([
    ('Montres', 'campaña de spam', 9, 8, 9),
    ('Colis en attente', 'campaña de spam', 5, 6, 4),
    ('Loterie', 'campaña de spam', 7, 7, 8),
    ('Lettre de la mairie', 'remitente legítimo', 4, 3, 5)
], columns=['group', 'kind', 'fold_1', 'fold_2', 'fold_3'])

# D6.E5 — la prueba A/B del teatro (ejercicio 6.5)
theater = pd.DataFrame([
    ('A', 'carta habitual', 1800, 126),
    ('B', 'carta con vídeo', 1750, 147)
], columns=['version', 'description', 'audience', 'bookings'])

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `message` | identificador del mensaje (M1…, R1…) | — | — |
| `label` | etiqueta real: spam o legítimo | — | `etiquette` |
| `links` | número de enlaces en el mensaje | enlaces | `liens` |
| `suspicious_words` | número de palabras sospechosas en el mensaje | palabras | `mots_suspects` |
| `messages` | número de mensajes | mensajes | — |
| `spams` | número de spams | mensajes | — |
| `subset` | conjunto: entrenamiento, validación o prueba | — | `lot` |
| `threshold` | umbral de decisión | — | `seuil` |
| `campaign` | campaña de envío del mensaje | — | `campagne` |
| `group` | grupo de mensajes (una campaña completa o un remitente), que se mantiene entero en un pliegue | — | `groupe` |
| `kind` | tipo de grupo: campaña de spam o remitente legítimo | — | `nature` |
| `fold` | número del pliegue de validación cruzada | — | `pli` |
| `fold_1` | D6.E3: número de mensajes del grupo en el pliegue 1 | mensajes | `pli_1` |
| `fold_2` | D6.E3: número de mensajes del grupo en el pliegue 2 | mensajes | `pli_2` |
| `fold_3` | D6.E3: número de mensajes del grupo en el pliegue 3 | mensajes | `pli_3` |
| `mean_links` | D6.4: media de los enlaces en los mensajes de entrenamiento de la ronda (los otros dos pliegues) | enlaces | `moyenne_liens` |
| `std_links` | D6.4: desviación típica de los enlaces en los mensajes de entrenamiento de la ronda (los otros dos pliegues) | enlaces | `ecart_type_liens` |
| `description` | descripción de la versión probada | — | — |
| `audience` | número de espectadores | personas | `spectateurs` |
| `bookings` | número de reservas | reservas | `reservations` |
| `customers` | número de clientes expuestos a la versión | clientes | `clients` |
| `orders` | número de pedidos | pedidos | `commandes` |
| `version` | versión de la campaña (A o B) | — | — |
| `twelve` | los doce mensajes del capítulo 5 | — | — |
| `loss_l001` | D6.4: pérdida de validación del pliegue, con λ = 0,01 | — | `perte_l001` |
| `loss_l01` | D6.4: pérdida de validación del pliegue, con λ = 0,1 | — | `perte_l01` |
| `fn` | número de falsos negativos (spams que se escapan) | mensajes | — |
| `fp` | número de falsos positivos (mensajes legítimos marcados) | mensajes | — |
| `split` | D6.2: división del conjunto de prueba (al azar o por campaña) | — | `decoupage` |
| `copies_seen` | D6.2: spams de prueba copiados de una campaña vista en el entrenamiento | mensajes | `copies_vues` |
| `missed_copies` | D6.2: copias que se escapan del filtro | mensajes | `manques_copies` |
| `missed_new` | D6.2: spams nuevos que se escapan del filtro | mensajes | `manques_nouveaux` |
| `lambda` | D6.E1: intensidad λ de la penalización | — | — |
| `loss` | D6.E1: pérdida de validación del pliegue para este λ | — | `perte` |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 6.1 a 6.6.

In [ ]:
# Lección 6.1 — coste por mensaje (ecuación 6.1), k = 3 para la asociación
k = 3
t = subsets.copy()
t['coste'] = t['fn'] + k * t['fp']
t['coste por mensaje'] = (t['coste'] / t['messages']).round(3)    # se divide por el tamaño DEL CONJUNTO
print(t.to_string(index=False))
val = t[t['subset'] == 'validation']
chosen = val.loc[val['coste por mensaje'].idxmin(), 'threshold']
print('umbral elegido en la validación:', chosen)
test = t[(t['subset'] == 'test') & (t['threshold'] == chosen)].iloc[0]
print(f"prueba, una sola vez: coste {test['coste']} sobre {test['messages']} mensajes = {test['coste por mensaje']:.3f} por mensaje")

a7 = t[t['threshold'] == 0.7]
colors = {'entrainement': SET_TRAIN, 'validation': SET_VAL, 'test': SET_TEST}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['subset'], a7['coste por mensaje'], color=[colors[l] for l in a7['subset']])
plt.ylabel('Coste por mensaje (umbral 0,7)'); plt.show()

In [ ]:
# Lección 6.2 — una fuga de datos: exhaustividad (recall) y coste por mensaje de las dos divisiones
for _, l in leak.iterrows():
    missed = l['missed_copies'] + l['missed_new']
    new_spams = l['spams'] - l['copies_seen']
    print(f"{l['split']:>12} : recall = {(l['spams'] - missed) / l['spams']:.2f}, "
          f"coste por mensaje = {(missed + 3 * l['fp']) / l['messages']:.3f}, "
          f"parte de spams nuevos no detectados = {l['missed_new']} / {new_spams} = {l['missed_new'] / new_spams:.2f}")

# Dividir las campañas, no los mensajes: pequeña demostración con 40 campañas de tamaño variable
rng = np.random.default_rng(6)
campaign = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # número de campaña de cada spam
at_random = rng.random(campaign.size) < 0.25                              # 25 % de los MENSAJES en la prueba
test_campaigns = rng.choice(40, size=10, replace=False)                         # 25 % de las CAMPAÑAS en la prueba
by_campaign = np.isin(campaign, test_campaigns)
for name, test_ in (('al azar', at_random), ('por campaña', by_campaign)):
    shared = len(set(campaign[test_]) & set(campaign[~test_]))
    print(f'{name:>12} : {test_.sum()} spams en la prueba, {shared} campañas presentes a la vez en la prueba y en otra parte')

In [ ]:
# Lección 6.3 — sobreajuste: filtros A y B en el entrenamiento (doce mensajes) y en la validación (seis)
X12, y12 = twelve[['links', 'suspicious_words']].to_numpy(float), (twelve['label'] == 'spam').to_numpy(int)
X6, y6 = six[['links', 'suspicious_words']].to_numpy(float), (six['label'] == 'spam').to_numpy(int)
filters = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for name, (w, b) in filters.items():
    p6 = sigmoid(X6 @ np.array(w) + b)
    losses6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filtro {name}: entrenamiento {mean_loss(y12, X12, w, b):.2f}, validación {mean_loss(y6, X6, w, b):.2f} '
          f'| pérdidas de los seis: {np.round(losses6, 2)}')

# Pérdida penalizada L2 (ecuación 6.2), λ = 0,05; la constante b no se penaliza
lam = 0.05
for name, (w, b) in filters.items():
    penalty = lam * np.sum(np.square(w))
    print(f'filtro {name}: suma de los cuadrados de los pesos = {np.sum(np.square(w)):.4f}, penalización = {penalty:.4f}, '
          f'pérdida penalizada = {mean_loss(y12, X12, w, b) + penalty:.2f}')

# La penalización depende de las unidades: enlaces contados en decenas, mismas predicciones, otra penalización
X12_tens = X12 / np.array([10, 1])
w_tens = (10, 0.5)
print('mismas probabilidades:', np.allclose(sigmoid(X12_tens @ np.array(w_tens) - 4), sigmoid(X12 @ np.array([1, 0.5]) - 4)),
      f'| penalización en unidades: {lam * (1 + 0.25):.4f}; en decenas: {lam * (100 + 0.25):.4f}')

In [ ]:
# Lección 6.4 — estandarización en cada vuelta: media y desviación típica de los enlaces en los OTROS DOS pliegues
for k in range(3):
    others = folds[folds['fold'] != k + 1]
    n = others['messages'].sum()
    m = (others['messages'] * others['mean_links']).sum() / n
    # desviación típica de la unión de dos pliegues (división por n): media de los cuadrados menos el cuadrado de la media
    v = (others['messages'] * (others['std_links'] ** 2 + others['mean_links'] ** 2)).sum() / n - m ** 2
    print(f'vuelta {k + 1}: validación = pliegue {k + 1}, entrenamiento = {n} mensajes, media {m:.2f}, desviación típica {np.sqrt(v):.2f}')
print(f'mensaje con 6 enlaces del pliegue 3, en la vuelta 3: (6 - 2,25) / 1,76 = {(6 - 2.25) / 1.76:.2f}')

# Puntuación de validación cruzada (ecuación 6.4) de los dos valores de λ
for col, lam in (('loss_l001', 0.01), ('loss_l01', 0.1)):
    print(f'λ = {lam}: pérdidas de los pliegues {list(folds[col])}, VC = {folds[col].mean():.3f}')

In [ ]:
# Lección 6.4 (final) — L1 y L2 sobre mensajes ficticios con ocho características (los mismos que en el ejercicio 6.4)
msgs = make_messages()
X8 = msgs[FEATURE_NAMES].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # aquí, estandarización sobre todos los mensajes: simple demostración
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = fit(X8, y8, 0.1)
w_l1, _ = fit(X8, y8, 0.1, l1=True)
print('pesos L2:', np.round(w_l2, 2))
print('pesos L1:', np.round(w_l1, 2))
print(f'pesos nulos: L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(las dos primeras características son las únicas relacionadas con la etiqueta)')

In [ ]:
# Lecciones 6.5 y 6.6 — la prueba A/B de la librería
a, b_ = bookshop.iloc[0], bookshop.iloc[1]
pA, pB = a['orders'] / a['customers'], b_['orders'] / b_['customers']
gap = pB - pA
print(f'tasa A = {100 * pA:.1f} %, tasa B = {100 * pB:.1f} %, diferencia = {100 * gap:.1f} puntos, '
      f'variación relativa = {100 * gap / pA:.0f} %')
seA, seB = np.sqrt(pA * (1 - pA) / a['customers']), np.sqrt(pB * (1 - pB) / b_['customers'])
se = np.sqrt(seA ** 2 + seB ** 2)                          # ecuación 6.6: se suman los CUADRADOS
low, high = gap - 1.96 * se, gap + 1.96 * se           # ecuación 6.7
print(f'EE de A = {100 * seA:.2f} punto, EE de B = {100 * seB:.2f} punto, EE de la diferencia = {100 * se:.2f} punto')
print(f'intervalo de confianza al 95 %: de {100 * low:.1f} a {100 * high:.1f} puntos')
print(f'para 40 000 suscriptores: {40000 * gap:.0f} pedidos más (de {40000 * low:.0f} a {40000 * high:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0: ningún efecto')
plt.axvline(1, color=THRESHOLD, linestyle='--', label='umbral de rentabilidad: 1 punto')
plt.plot([100 * low, 100 * high], [0, 0], color=RESIDUAL, linewidth=4, label='intervalo al 95 %')
plt.scatter([100 * gap], [0], marker='D', s=70, color=RESIDUAL, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Efecto de la versión B (puntos porcentuales)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Lección 6.6 — el coste por mensaje de la prueba (lección 6.1) es una media: mismo razonamiento
costs = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 spams no detectados, 4 falsas alarmas, 226 sin coste
se_c = costs.std() / np.sqrt(len(costs))                     # desviación típica (división por n) / raíz de n
print(f'coste por mensaje = {costs.mean():.3f}, desviación típica = {costs.std():.3f}, EE = {se_c:.3f}, '
      f'intervalo: de {costs.mean() - 1.96 * se_c:.2f} a {costs.mean() + 1.96 * se_c:.2f}')

## Ejercicio 6.1 — Leer un gráfico: tres pliegues, cinco valores de λ

*Conceptos: validación cruzada (lección 6.4, ecuación 6.4), sobreajuste y penalización (lección 6.3).* Datos D6.E1: la pérdida de validación de cada pliegue de la mediateca, para cinco valores de $\lambda$.

### a) a e) En papel, leyendo la figura del libro

*Tus lecturas y tus respuestas:* …

In [ ]:
# PARA COMPLETAR: media de los tres pliegues para cada λ (ecuación 6.4), luego el λ elegido
means = None   # PARA COMPLETAR: una Series indexada por λ; prueba folds_lambda.groupby('lambda')['loss']
lam_chosen = None   # PARA COMPLETAR

In [ ]:
# Comprobación de a) y b)
if means is None or lam_chosen is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    print('medias:', means.round(3).to_dict(), '→',
          'correcto' if np.allclose(means.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'revisar')
    print('λ elegido:', lam_chosen, '→', 'correcto' if lam_chosen == 0.1 else 'revisar')
    print('elección de cada pliegue por separado:', folds_lambda.loc[folds_lambda.groupby('fold')['loss'].idxmin(), ['fold', 'lambda']].values.tolist())
    for k, (style, marker) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = folds_lambda[folds_lambda['fold'] == k + 1]
        plt.plot(d['lambda'], d['loss'], style, marker=marker, color=SET_VAL, label=f'pliegue {k + 1}')
    plt.plot(means.index, means.values, 'D-', color='black', linewidth=2.5, label='media')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Pérdida de validación'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Ejercicio 6.2 — Elegir y después evaluar un umbral: la mediateca

*Conceptos: tres conjuntos (lección 6.1), coste por mensaje (ecuación 6.1), coste de un umbral (ecuación 5.9).* Para la mediateca, una falsa alarma cuesta lo mismo que dos spams no detectados: $k = 2$. Datos D6.E2. Haz primero las preguntas a) a e) en papel, **sin mirar las filas de la prueba antes de la pregunta b)**.

### a) a e) En papel

*Tus cálculos y tus respuestas:* …

In [ ]:
# PARA COMPLETAR: coste por mensaje (ecuación 6.1) y elección del umbral en la VALIDACIÓN
def cost_per_message(fn, fp, n, k):
    return None   # PARA COMPLETAR


def threshold_choice(validation_table, k):
    """Devuelve el umbral cuyo coste por mensaje es el más pequeño en la validación."""
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación de a) y b), y después el protocolo
k_med = 2
valid = media_library[media_library['subset'] == 'validation']
test_med = media_library[media_library['subset'] == 'test']
if cost_per_message(3, 18, 150, 2) is None or threshold_choice(valid, k_med) is None:
    print('aún sin completar. Completa las funciones.')
else:
    cv = [cost_per_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validación, coste por mensaje:', np.round(cv, 3), '→',
          'correcto' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'revisar')
    s = threshold_choice(valid, k_med)
    print(f'umbral elegido en la validación: {s} →', 'correcto' if s == 0.6 else 'revisar')
    l = test_med[test_med['threshold'] == s].iloc[0]                    # UNA sola fila de la prueba
    ct = cost_per_message(l.fn, l.fp, l.messages, k_med)
    print(f'prueba, solo con el umbral {s}: {ct:.3f} por mensaje →', 'correcto' if abs(ct - 35 / 240) < 1e-9 else 'revisar')
    # e) Lo que hace el colega: mira TODAS las filas de la prueba y se queda con la mejor
    ct_all = [cost_per_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['threshold'].iloc[int(np.argmin(ct_all))]
    print(f'anuncio del colega: umbral {s_coll}, {min(ct_all):.3f} por mensaje (elegido en la prueba: demasiado optimista);',
          f'anuncio honesto: umbral {s}, {ct:.3f} por mensaje →',
          'correcto' if s_coll == 0.7 and abs(min(ct_all) - 28 / 240) < 1e-9 else 'revisar')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['threshold'], cv, 'o-', color=SET_VAL, label='validación (sirve para elegir)')
    plt.plot(test_med['threshold'], ct_all, 's--', color=SET_TEST, alpha=0.5, label='prueba (no mirarla para elegir)')
    plt.scatter([s], [ct], s=120, color=SET_TEST, zorder=3, label='prueba, con el umbral elegido')
    plt.xlabel('Umbral'); plt.ylabel('Coste por mensaje (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Tu conclusión

*Lo que la mediateca puede anunciar:* …

## Ejercicio 6.3 — Interpretar: una fuga entre los pliegues

*Conceptos: fuga de datos (lección 6.2), validación cruzada por grupos y estandarización en cada vuelta (lección 6.4).* Datos D6.E3: dónde cayeron los mensajes de cuatro grupos en los pliegues del voluntario.

### a) a d) En papel

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: grupos presentes en más de un pliegue
def split_groups(allocation):
    """allocation: una tabla con una columna 'group' y las columnas 'fold_1', 'fold_2', 'fold_3'
    (número de mensajes del grupo en cada pliegue). Devuelve la lista de los grupos divididos."""
    return None   # PARA COMPLETAR: cuenta, para cada grupo, los pliegues en los que tiene al menos un mensaje

In [ ]:
# Comprobación de b) y d)
proposal = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['group', 'fold_1', 'fold_2', 'fold_3'])
if split_groups(folds_of_groups) is None:
    print('aún sin completar. Completa la función.')
else:
    print('pliegues del voluntario, grupos divididos:', split_groups(folds_of_groups), '→',
          'correcto' if len(split_groups(folds_of_groups)) == 4 else 'revisar')
    print('reparto corregido, grupos divididos:', split_groups(proposal), '| tamaños de los pliegues:',
          proposal[['fold_1', 'fold_2', 'fold_3']].sum().tolist(), '→',
          'correcto' if split_groups(proposal) == [] else 'revisar')
    print('vuelta 1: copias de « Montres » vistas en el entrenamiento =', int(folds_of_groups.loc[0, ['fold_2', 'fold_3']].sum()),
          ', en la validación =', int(folds_of_groups.loc[0, 'fold_1']))

## Ejercicio 6.4 — Validación cruzada por campañas

*Conceptos: validación cruzada (lección 6.4, ecuación 6.4), penalización L2 (ecuación 6.2), estandarización (ecuación 2.7), fuga (lección 6.2).* Los mensajes los genera `make_messages()` (celda de importación). Los grupos 30 a 39 forman el conjunto de prueba: no se tocan hasta la pregunta e).

In [ ]:
# Los mensajes del ejercicio: 30 grupos para la validación cruzada, 10 grupos apartados para la prueba
msgs = make_messages()
test_ = msgs['group'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, FEATURE_NAMES].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'group'].to_numpy()
X_test, y_test = msgs.loc[test_, FEATURE_NAMES].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} mensajes, {msgs.group.nunique()} grupos; validación cruzada: {len(y_cv)} mensajes; prueba: {len(y_test)} mensajes')
print(msgs[FEATURE_NAMES].describe().loc[['mean', 'std']].round(1))     # unidades muy distintas

### a) Tu predicción

*Cuando λ pasa de 0 a 1, la pérdida de validación cruzada…* …

In [ ]:
# PARA COMPLETAR: b) pliegues por grupo, estandarización sobre el entrenamiento, c) puntuación de validación cruzada
def group_folds(groups, K):
    """Número de pliegue (0 a K - 1) de cada mensaje: los grupos, ordenados de menor a mayor, se reparten
    por turnos en los pliegues; todos los mensajes de un grupo reciben el mismo pliegue."""
    return None   # PARA COMPLETAR; np.unique(groups) da los grupos ordenados


def standardization(X_train, X_val):
    """Estandariza X_train y X_val con la media y la desviación típica de X_train SOLAMENTE."""
    return None, None   # PARA COMPLETAR


def cv_score(lam, folds_, K=3):
    """Ecuación 6.4: media, sobre las K vueltas, de la pérdida SIN penalización en el pliegue de validación."""
    return None   # PARA COMPLETAR: usa standardization, fit(X, y, lam) y mean_loss(y, X, w, b)

In [ ]:
# Comprobación de b) y c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = group_folds(g_cv, 3)
if p_g is None or standardization(X_cv, X_cv)[0] is None or cv_score(0.1, p_g) is None:
    print('aún sin completar. Completa las tres funciones.')
else:
    splits = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'pliegues por grupo: {np.bincount(p_g).tolist()} mensajes, {splits} grupo(s) dividido(s) →',
          'correcto' if splits == 0 and set(p_g) == {0, 1, 2} else 'revisar')
    Xe, Xv = standardization(X_cv[p_g != 0], X_cv[p_g == 0])
    print('entrenamiento estandarizado: medias ≈ 0 y desviaciones típicas ≈ 1; validación: no exactamente →',
          'correcto' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'revisar')
    cv_group = {lam: cv_score(lam, p_g) for lam in LAMBDAS}
    lam_cv = min(cv_group, key=cv_group.get)
    print('VC por grupo:', {l: round(v, 3) for l, v in cv_group.items()}, '→ λ elegido:', lam_cv, '→',
          'correcto' if lam_cv == 0.01 else 'revisar')

In [ ]:
# d) Pliegues sorteados mensaje a mensaje (no hacerlo): la fuga de la lección 6.2, tres veces
if p_g is not None and cv_score(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    cv_message = {lam: cv_score(lam, p_m) for lam in LAMBDAS}
    print('VC mensaje a mensaje:', {l: round(v, 3) for l, v in cv_message.items()})
    plt.plot(range(5), list(cv_group.values()), 'o-', color=SET_VAL, label='pliegues por grupo')
    plt.plot(range(5), list(cv_message.values()), 's--', color='grey', label='pliegues mensaje a mensaje (fuga)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Puntuación de validación cruzada')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Modelo final: estandarización y pesos sobre los 30 grupos, con el λ elegido; prueba UNA sola vez
    Xc, Xt = standardization(X_cv, X_test)
    w_f, b_f = fit(Xc, y_cv, lam_cv)
    print(f'pérdida de prueba del modelo final (λ = {lam_cv}): {mean_loss(y_test, Xt, w_f, b_f):.3f}; '
          f'VC por grupo: {cv_group[lam_cv]:.3f}; VC mensaje a mensaje: {cv_message[lam_cv]:.3f}')

### d) y e) Tu interpretación

…

## Ejercicio 6.5 — Predecir y después comprobar: el recordatorio del teatro

*Conceptos: diferencia en puntos (ecuación 6.5), error estándar (ecuación 6.6), intervalo de confianza (ecuación 6.7).* Datos D6.E5. El vídeo solo merece la pena si hace ganar al menos 0,5 puntos.

### a) a e) En papel

*Tus cálculos, tu predicción y tus respuestas:* …

In [ ]:
# PARA COMPLETAR: diferencia en puntos e intervalo de confianza al 95 % (ecuaciones 6.5 a 6.7)
def interval(conv_A, n_A, conv_B, n_B):
    """Devuelve (diferencia, límite inferior, límite superior), en puntos porcentuales."""
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación de a), b) y e)
A, B = theater.iloc[0], theater.iloc[1]
r = interval(A['bookings'], A['audience'], B['bookings'], B['audience'])
if r is None:
    print('aún sin completar. Completa la función.')
else:
    print(f'ensayo real: diferencia {r[0]:.2f} puntos, intervalo de {r[1]:.2f} a {r[2]:.2f} →',
          'correcto' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'revisar')
    r4 = interval(4 * A['bookings'], 4 * A['audience'], 4 * B['bookings'], 4 * B['audience'])
    print(f'ensayo cuatro veces más grande: de {r4[1]:.2f} a {r4[2]:.2f}; anchura {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correcto' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'revisar')

In [ ]:
# f) 1000 ensayos simulados cuyo efecto verdadero vale 1,4 puntos: ¿cuántos intervalos contienen 1,4?
rng = np.random.default_rng(65)
nA, nB, true_A, true_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, true_A, 1000) / nA
b = rng.binomial(nB, true_B, 1000) / nB
se_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contains = np.abs((b - a) - 0.014) <= 1.96 * se_sim
print(f'proporción de intervalos que contienen el efecto verdadero: {contains.mean():.1%}')
print(f'proporción de intervalos que contienen el cero: {np.mean(np.abs(b - a) <= 1.96 * se_sim):.1%}')

### f) Tu conclusión

…

## Ejercicio 6.6 — Desafío: ¿cuántos clientes, y cuándo mirar?

*Conceptos: error estándar e intervalo de confianza (lección 6.6), distribución binomial (lección 4.5).*

### a) a c) En papel

*Tus cálculos y tu predicción:* …

In [ ]:
# PARA COMPLETAR: a) tamaño por grupo para una semianchura dada (en puntos)
def group_size(pA, pB, half_width):
    """pA, pB en porcentaje; devuelve el menor n entero tal que 1,96 × EE ≤ half_width."""
    return None   # PARA COMPLETAR; np.ceil redondea al entero superior

In [ ]:
# Comprobación de a) y b)
n1 = group_size(8, 10, 1)
if n1 is None:
    print('aún sin completar. Completa la función.')
else:
    print(f'semianchura 1 punto: {n1} clientes por grupo →', 'correcto' if n1 == 6285 else 'revisar')
    print(f'semianchura 0,5 punto: {group_size(8, 10, 0.5)} clientes por grupo, '
          f'{2 * group_size(8, 10, 0.5)} en total')

In [ ]:
# d) 2000 ensayos SIN efecto. Calendario: 250 clientes por día y por grupo, 20 lecturas (una cada tarde,
#    sobre los clientes acumulados: 250, 500… 5000 por grupo). Regla de parada: primera tarde en que el intervalo excluye el 0.
rng = np.random.default_rng(66)
trials, days, per_day, p = 2000, 20, 250, 0.08
cA = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # pedidos acumulados, grupo A
cB = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # grupo B: misma tasa, ningún efecto
n = per_day * np.arange(1, days + 1)                                  # clientes por grupo al final de cada día
pA_, pB_ = cA / n, cB / n
se_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
excludes = np.abs(pB_ - pA_) > 1.96 * se_j                                # ¿excluye el cero el intervalo de la tarde?
end, one_evening = excludes[:, -1].mean(), excludes.any(axis=1).mean()
print(f'intervalo leído solo el día 20: {end:.1%} de los ensayos concluyen erróneamente que hay un efecto')
print(f'intervalo leído cada tarde, parada en el primer « éxito »: {one_evening:.1%}')

### d) y e) Tu explicación

…